In [6]:
import heapq
from sortedcontainers import SortedList

# Helper functions for segment intersection
def orientation(p, q, r):
    val = (q[1] - p[1]) * (r[0] - q[0]) - \
          (q[0] - p[0]) * (r[1] - q[1])
    if val == 0: return 0  # Collinear
    return 1 if val > 0 else 2 # Clockwise or Counterclockwise

def on_segment(p, q, r):
    if (q[0] <= max(p[0], r[0]) and q[0] >= min(p[0], r[0]) and
            q[1] <= max(p[1], r[1]) and q[1] >= min(p[1], r[1])):
        return True
    return False

def segments_intersect(p1, q1, p2, q2):
    o1 = orientation(p1, q1, p2)
    o2 = orientation(p1, q1, q2)
    o3 = orientation(p2, q2, p1)
    o4 = orientation(p2, q2, q1)

    # General case
    if o1 != 0 and o2 != 0 and o3 != 0 and o4 != 0 and \
       o1 != o2 and o3 != o4:
        return True

    # Special Cases
    # p1, q1 and p2 are collinear and p2 lies on segment p1q1
    if o1 == 0 and on_segment(p1, p2, q1): return True

    # p1, q1 and q2 are collinear and q2 lies on segment p1q1
    if o2 == 0 and on_segment(p1, q2, q1): return True

    # p2, q2 and p1 are collinear and p1 lies on segment p2q2
    if o3 == 0 and on_segment(p2, p1, q2): return True

    # p2, q2 and q1 are collinear and q1 lies on segment p2q2
    if o4 == 0 and on_segment(p2, q1, q2): return True

    return False # Doesn't fall in any of the above cases

# Assuming segment_intersect is an alias for segments_intersect
def segment_intersect(p1, q1, p2, q2):
    return segments_intersect(p1, q1, p2, q2)

def intersection_point(p1, q1, p2, q2):
    # Using line-line intersection formula
    # Line AB is p1 to q1, Line CD is p2 to q2
    # A = (x1, y1), B = (x2, y2)
    # C = (x3, y3), D = (x4, y4)

    x1, y1 = p1
    x2, y2 = q1
    x3, y3 = p2
    x4, y4 = q2

    den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if den == 0: # Lines are parallel or collinear
        return None

    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / den
    u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / den

    # If 0 <= t <= 1 and 0 <= u <= 1, then segments intersect
    if 0 <= t <= 1 and 0 <= u <= 1:
        intersect_x = x1 + t * (x2 - x1)
        intersect_y = y1 + t * (y2 - y1)
        return (intersect_x, intersect_y)
    return None


def sweep_line_intersect(segments):
    events = []
    for i, (p,q) in enumerate(segments):
        if p[0] > q[0]: p, q = q, p
        heapq.heappush(events, (p[0], 0, i, p ,q))
        heapq.heappush(events, (q[0], 1, i, p, q))

    active = SortedList(key = lambda s: s[0][1])

    while events:
        x, kind, i, p, q = heapq.heappop(events)

        if kind == 0:
            active.add((p,q,i))
            pos = active.index((p,q,i))

            if pos > 0:
                s = active[pos - 1]
                # segments_intersect needs to be defined
                if segments_intersect(p,q, s[0],s[1]):
                    return True
            if pos < len(active) - 1:
                s = active[pos + 1]
                # segment_intersect needs to be defined
                if segment_intersect(p,q,s[0],s[1]):
                    return True

        else:
            try:
                pos = active.index((p,q,i))
                if 0 < pos < len(active) - 1:
                    lo, hi = active[pos - 1], active[pos + 1]
                    # segments_intersect needs to be defined
                    if segments_intersect(lo[0], lo[1], hi[0],hi[1]):
                        return True
                active.remove((p,q,i))
            except ValueError:
                pass

    return False


def brute_force_intersect(segments):
    n = len(segments)
    for i in range(n):
        for j in range(i + 1, n):
            a1, a2 = segments[i]; b1,b2 = segments[j]
            # segments_interesect (typo, should be segments_intersect) and intersection_point needs to be defined
            if segments_intersect(a1,a2,b1,b2): # Corrected typo here
                yield(i,j, intersection_point(a1,a2,b1,b2))


segs = [
    ((0,0),(4,4)), ((0,4),(4,0)),
    ((5,0),(5,4)), ((3,1),(7,3)),
]

# Changed 'point' to 'print'
print(sweep_line_intersect(segs))
for pair in brute_force_intersect(segs):
    print(pair)


True
(0, 1, (2.0, 2.0))
(1, 3, (3.0, 1.0))
(2, 3, (5.0, 2.0))


In [3]:
!pip install sortedcontainers

In [8]:
import numpy as np

def aabb_intersect(a,b):
    return not (a[2] < b[0] or
                b[2] < a[0] or
                a[3] < b[1] or
                b[3] < a[1])


def aabb_overlap_area(a,b):
    ox = max(0, min(a[2],b[2]) - max(a[0],b[0]))
    oy = max(0, min(a[3],b[3]) - max(a[1],b[1]))

    return ox*oy


def sort_and_sweep(boxes):
    events = []
    for i, (x0,y0,x1,y1) in enumerate(boxes):
        events.append((x0, 0, i))
        events.append((x1, 1 , i))

    events.sort()



    active = []
    pairs = []

    for x, kind, i in events:
        if kind == 0:
            for j in active:
                if aabb_intersect(boxes[i], boxes[j]):
                    pairs.append((i,j))
            active.append(i)
        else:
            active.remove(i)
    return pairs



def aabb_all_pairs_np(boxes):
    b = np.array(boxes)
    x0,y0,x1,y1 = b[:,0],b[:,1],b[:,2],b[:,3]

    no_overlap = ((x1[:,None] < x0[None,:]) |
                 (x0[:,None]  > x1[None,:]) |
                 (y1[:,None] < y0[None,:])  |
                 (y0[:,None] > y1[None, :]))
    mask = ~no_overlap
    np.fill_diagonal(mask, False)
    return np.argwhere(np.triu(mask))



boxes = [(0,0,4,4), (2,2,6,6), (5,5,9,9), (8,0,12,4)]
print(sort_and_sweep(boxes))
print(aabb_intersect(boxes[0], boxes[1]))
print(aabb_overlap_area(boxes[0], boxes[1]))


[(1, 0), (2, 1)]
True
4


In [10]:
import numpy as np


def project_polygon(polygon, axis):
    dots = [np.dot(v,axis) for v in polygon]
    return min(dots), max(dots)

def get_axes(polygon):
    axes = []
    n = len(polygon)
    for i in range(n):
        edge = np.array(polygon[(i + 1) % n]) - np.array(polygon[i])
        normal = np.array([-edge[1], edge[0]])
        length = np.linalg.norm(normal)
        if length > 1e-10:
            axes.append(normal / length)
    return axes



def sat_intersect(poly_a, poly_b):
    axes = get_axes(poly_a) + get_axes(poly_b)
    for axis in axes:
        min_a, max_a = project_polygon(poly_a, axis)
        min_b, max_b = project_polygon(poly_b, axis)
        if max_a < min_b or max_b < min_a:
            return False
    return True


def sat_mtv(poly_a, poly_b):
    axes = get_axes(poly_a) + get_axes(poly_b)
    min_overlap, best_axis = float('inf'), None

    for axis in axes:
        min_a, max_a = project_polygon(poly_a, axis)
        min_b, max_b = project_polygon(poly_b, axis)
        overlap = min(max_a, max_b) - max(min_a, min_b)
        if overlap < 0: return None, None
        if overlap < min_overlap:
            min_overlap, best_axis = overlap, axis

    mtv = best_axis * min_overlap
    return True, mtv


square = [(0,0), (4,0), (4,4), (0,4)]
diamon = [(2,0), (5,3), (2,6), (-1,3)]
print(sat_intersect(square, diamon))

far = [(10,10),(14,10),(14,14),(10,14)]
print(sat_intersect(square, far))

hit, mtv = sat_mtv(square, diamon)
print(f"MTV: {mtv}")


True
False
MTV: [0. 4.]


In [12]:
import numpy as np

def circle_circle(c1,r1,c2,r2):
    d = np.linalg.norm(np.array(c2) - np.array(c1))
    if d < abs(r1 - r2): return 'contained'
    if d <= r1 + r2:     return 'intersect'
    return 'disjoint'


def segment_circle(p1, p2, center, radius):
    p1, p2, c= np.array(p1), np.array(p2), np.array(center)
    d = p2 - p1
    f = p1 - c
    a = np.dot(d,d)
    b = 2 * np.dot(f,d)
    c_ = np.dot(f,f) - radius**2
    disc = b**2 - 4*a*c_
    if disc < 0: return False
    disc_sqrt = disc**0.5
    t1 = (-b - disc_sqrt) / (2*a)
    t2 = (-b + disc_sqrt) / (2*a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


def point_in_circle(point, center, radius):
    d =  np.linalg.norm(np.array(point) - np.array(center))
    return d <= radius


def aabb_circle(box, center, radius):
    cx, cy = center
    nearest_x = max(box[0], min(cx, box[2]))
    nearest_y = max(box[1], min(cy, box[3]))
    dist = ((cx - nearest_x) ** 2 + (cy - nearest_y)**2)**0.5
    return dist <= radius


def circles_sweep(circles):
    events = []
    for i, (cx,cy,r) in enumerate(circles):
        events += [(cx-r, 0 , i), (cx + r, 1, i)]
    events.sort()
    active, pairs = [], []
    for _, kind, i in events:
        if kind == 0:
            for j in active:
                c1, r1 = circles[i][:2], circles[i][2]
                c2,r2 = circles[j][:2], circles[j][2]
                if circle_circle(c1,r1,c2,r2) != 'disjoint':
                    pairs.append((i,j))
            active.append(i)
        else:
            active.remove(i)
    return pairs


print(circle_circle((0,0),3,(4,0),2))
print(segment_circle((0,0),(10,0),(5,2),3))
print(aabb_circle((0,0,4,4), (6,2),3))


intersect
True
True


In [ ]:
import bisect

def next_fit(items, capacity):
    bins = [[]]
    remaining = [capacity]
    for item in items:
        if item > remaining[-1]:
            bins.append([])
            remaining.append(capacity)
        bins[-1].append(item)
        remaining[-1] -= item
    return bins


def first_fit(items, capacity):
    bins = []
    remaining = []
    for item in items:
        placed = False
        for i, rem in enumerate(remaining):
            if rem >= item:
                bins[i].append(item)
                remaining[i] -= item
                placed = True
                break
        if not placed:
            bins.append([item])
            remaining.append(capacity - item)
    return bins


def best_fit(items, capacity):
    bins = []
    for item in items:
        best_idx, best_rem = -1, capacity + 1
        for i, rem in enumerate(remaining):
            if rem >= item and rem - item < best_rem:
                best_idx, best_rem = i, rem - item
        if best_idx == -1:
            bins.append([item]); remaining.append(capacity - item)
        else:
            bins[best_idx]